In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.optim import AdamW
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve, auc
from peft import LoraConfig, get_peft_model  # For efficient fine-tuning



e:\ML\BioActivity\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
test_df=pd.read_excel('bioactivity_dataset_cleaned_outliers.xlsx')
test_df.head()

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class,bioactivity
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,464.522,3.9242,4,6,8.6,active,1
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,485.544,4.4323,4,6,9.0,active,1
2,O=C(/C=C/c1cccc(C(C(=O)Nc2ccccc2)C(=O)Nc2ccccc...,415.449,3.5662,4,4,9.0,active,1
3,O=C(CCCCCCC(=O)Nc1ccccc1)NO,264.325,2.4711,3,3,6.7,active,1
4,O=C(CCCCCC/C(=N\O)c1ccc(-c2ccccc2)cc1)NO,340.423,4.3779,3,4,8.4,active,1


In [ ]:
df = pd.read_excel('bioactivity_dataset_cleaned_outliers.xlsx')
df['mol'] = df['canonical_smiles'].apply(Chem.MolFromSmiles)
df = df.dropna(subset=['mol'])

# Compute/scale descriptors (if not already)
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(4)
    return np.array([Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
                     rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol)])

df[['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']] = df['canonical_smiles'].apply(compute_descriptors).tolist()

scaler = StandardScaler()
desc_cols = ['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']
df[desc_cols] = scaler.fit_transform(df[desc_cols])

In [27]:
t_df = pd.read_excel('bioactivity_dataset_cleaned_outliers.xlsx')
t_df['mol'] = t_df['canonical_smiles'].apply(Chem.MolFromSmiles)
t_df = t_df.dropna(subset=['mol'])

# Compute/scale descriptors (if not already)
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return np.zeros(4)
    return np.array([Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
                     rdMolDescriptors.CalcNumHBD(mol), rdMolDescriptors.CalcNumHBA(mol)])

t_df[['MW', 'LogP', 'NumHDonors', 'NumHAcceptors']] = t_df['canonical_smiles'].apply(compute_descriptors).tolist()
t_df.head(5)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class,bioactivity,mol
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,464.522,3.9242,4.0,6.0,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B5926...
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,485.544,4.4323,4.0,6.0,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B599B...
2,O=C(/C=C/c1cccc(C(C(=O)Nc2ccccc2)C(=O)Nc2ccccc...,415.449,3.5662,4.0,4.0,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B599B...
3,O=C(CCCCCCC(=O)Nc1ccccc1)NO,264.325,2.4711,3.0,3.0,6.7,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B599B...
4,O=C(CCCCCC/C(=N\O)c1ccc(-c2ccccc2)cc1)NO,340.423,4.3779,3.0,4.0,8.4,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B599B...


In [32]:
t_df['canonical_smiles'].iloc[1]

'O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2ncccc2c1)NO'

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5667 entries, 0 to 5666
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   canonical_smiles   5667 non-null   object 
 1   MW                 5667 non-null   float64
 2   LogP               5667 non-null   float64
 3   NumHDonors         5667 non-null   float64
 4   NumHAcceptors      5667 non-null   float64
 5   pIC50              5667 non-null   float64
 6   bioactivity_class  5667 non-null   object 
 7   bioactivity        5667 non-null   int64  
 8   mol                5667 non-null   object 
dtypes: float64(5), int64(1), object(3)
memory usage: 398.6+ KB


In [4]:
df.head(3)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class,bioactivity,mol
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,0.524109,0.241522,1.32487,0.282913,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...
1,O=C(CCCCCC(C(=O)Nc1ccc2ncccc2c1)C(=O)Nc1ccc2nc...,0.732961,0.621072,1.32487,0.282913,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...
2,O=C(/C=C/c1cccc(C(C(=O)Nc2ccccc2)C(=O)Nc2ccccc...,0.036571,-0.025903,1.32487,-0.713216,9.0,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...


In [5]:
# Augmentation (3 variants for ~15k samples)
def augment_smiles(smiles, num_aug=3):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return [smiles]
    variants = [Chem.MolToSmiles(mol, canonical=True)]
    for _ in range(num_aug):
        try:
            variant = Chem.MolToSmiles(mol, canonical=False, doRandom=True)
            if Chem.MolFromSmiles(variant) is not None:
                variants.append(variant)
        except: pass
    return variants

aug_df = []
for _, row in df.iterrows():
    for var_smiles in augment_smiles(row['canonical_smiles']):
        aug_row = row.copy()
        aug_row['canonical_smiles'] = var_smiles
        aug_df.append(aug_row)
df = pd.DataFrame(aug_df)

In [6]:
df.head(3)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,pIC50,bioactivity_class,bioactivity,mol
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,0.524109,0.241522,1.32487,0.282913,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...
0,n1c2c(NC(=O)C(CCCCCC(=O)NO)NC(=O)OCc3ccccc3)cc...,0.524109,0.241522,1.32487,0.282913,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...
0,n1c2c(ccc1)cccc2NC(C(NC(=O)OCc1ccccc1)CCCCCC(=...,0.524109,0.241522,1.32487,0.282913,8.6,active,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...


In [23]:
df['canonical_smiles'].iloc[0]

'O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc12)NO'

In [7]:
df.drop(columns=['pIC50', 'bioactivity_class'], inplace=True)

In [8]:
df.head(2)

,canonical_smiles,MW,LogP,NumHDonors,NumHAcceptors,bioactivity,mol
0,O=C(CCCCCC(NC(=O)OCc1ccccc1)C(=O)Nc1cccc2cccnc...,0.524109,0.241522,1.32487,0.282913,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...
0,n1c2c(NC(=O)C(CCCCCC(=O)NO)NC(=O)OCc3ccccc3)cc...,0.524109,0.241522,1.32487,0.282913,1,<rdkit.Chem.rdchem.Mol object at 0x000001B4B9D...


In [9]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['bioactivity'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['bioactivity'], random_state=42)
tokenizer = AutoTokenizer.from_pretrained('DeepChem/ChemBERTa-77M-MTR')

class BioactivityDataset(Dataset):
    def __init__(self, df, tokenizer, desc_scaler):
        self.smiles = df['canonical_smiles'].tolist()
        self.labels = df['bioactivity'].tolist()
        self.descs = df[desc_cols].values
        self.desc_scaler = desc_scaler
        self.encodings = tokenizer(self.smiles, truncation=True, padding=True, max_length=512, return_tensors='pt')
    
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        item['descriptors'] = torch.tensor(self.descs[idx], dtype=torch.float)
        return item

In [10]:
train_dataset = BioactivityDataset(train_df, tokenizer, scaler)
val_dataset = BioactivityDataset(val_df, tokenizer, scaler)
test_dataset = BioactivityDataset(test_df, tokenizer, scaler)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [11]:
train_dataset[0]

{'input_ids': tensor([12, 16, 15, 20, 15, 17, 16, 16, 17, 23, 16, 17, 15, 21, 25, 15, 17, 15,
         25, 21, 18, 31, 15, 21, 15, 15, 15, 26, 15, 15, 15, 15, 15, 26, 15, 21,
         18, 16, 16, 16, 16, 16, 16, 17, 22, 19, 18, 19, 18, 22, 19, 18, 15, 21,
         15, 15, 17, 15, 15, 15, 21, 25, 20, 18, 19, 16, 13,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [12]:
train_df['bioactivity'].value_counts(), val_df['bioactivity'].value_counts(), test_df['bioactivity'].value_counts()

(bioactivity
 1    14851
 0     3283
 Name: count, dtype: int64,
 bioactivity
 1    1857
 0     410
 Name: count, dtype: int64,
 bioactivity
 1    1856
 0     411
 Name: count, dtype: int64)

In [13]:
class FineTunedMultimodalModel(nn.Module):
    def __init__(self, num_desc=4, emb_dim=384, hidden_dim=256, dropout=0.4, num_layers_to_unfreeze=2):
        super().__init__()
        self.smiles_encoder = AutoModel.from_pretrained('DeepChem/ChemBERTa-77M-MTR')

        self.config = self.smiles_encoder.config
        self.config.num_labels = 1  # Optional: Set for clas    sification task
        
        # Freeze all except last num_layers_to_unfreeze
        for param in self.smiles_encoder.parameters():
            param.requires_grad = False
        unfreeze_modules = self.smiles_encoder.encoder.layer[-num_layers_to_unfreeze:]
        for module in unfreeze_modules:
            for param in module.parameters():
                param.requires_grad = True
        
        self.desc_proj = nn.Linear(num_desc, emb_dim)
        self.fusion = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, input_ids=None, attention_mask=None, descriptors=None, **kwargs):
        outputs = self.smiles_encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        smiles_emb = outputs.last_hidden_state[:, 0, :]
        desc_emb = self.desc_proj(descriptors)
        fused = torch.cat([smiles_emb, desc_emb], dim=-1)
        return self.fusion(fused).squeeze(-1)

model = FineTunedMultimodalModel(num_layers_to_unfreeze=1)  # Start with 2 for small data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FineTunedMultimodalModel(
  (smiles_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(600, 384, padding_idx=1)
      (position_embeddings): Embedding(515, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.144, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-2): 3 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.109, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
          

In [136]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig( task_type=TaskType.SEQ_CLS, r=4, lora_alpha=8, target_modules=["query", "value"], lora_dropout=0.1)

In [137]:
model = get_peft_model(model, lora_config)
 
# Print trainable params (should be <1% of total)
model.print_trainable_parameters()  

trainable params: 18,432 || all params: 3,677,681 || trainable%: 0.5012


In [14]:
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(train_df['bioactivity']), y=train_df['bioactivity'])
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)  # Low LR for fine-tuning
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

criterion = BCEWithLogitsLoss(pos_weight=class_weights[1])
epochs = 50  # Longer for fine-tuning
best_auc = 0
patience, counter = 3, 0

In [15]:
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        descriptors = batch['descriptors'].to(device)
        labels = batch['labels'].float().to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, descriptors)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step(train_loss / len(train_loader))  # Adjust LR
    
    # Validation
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            descriptors = batch['descriptors'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attention_mask, descriptors)
            probs = torch.sigmoid(logits).cpu().numpy()
            val_preds.extend(probs)
            val_labels.extend(labels.cpu().numpy())
    
    auc = roc_auc_score(val_labels, val_preds)
    print(f'Epoch {epoch+1}: Train Loss {train_loss/len(train_loader):.4f}, Val AUROC {auc:.4f}')
    
    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), 'best_finetuned_model.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping!")
            break

Epoch 1: Train Loss 0.2987, Val AUROC 0.8968
Epoch 2: Train Loss 0.2232, Val AUROC 0.9370
Epoch 3: Train Loss 0.1857, Val AUROC 0.9517
Epoch 4: Train Loss 0.1607, Val AUROC 0.9603
Epoch 5: Train Loss 0.1447, Val AUROC 0.9678
Epoch 6: Train Loss 0.1300, Val AUROC 0.9713
Epoch 7: Train Loss 0.1207, Val AUROC 0.9730
Epoch 8: Train Loss 0.1085, Val AUROC 0.9775
Epoch 9: Train Loss 0.0978, Val AUROC 0.9809
Epoch 10: Train Loss 0.0917, Val AUROC 0.9779
Epoch 11: Train Loss 0.0851, Val AUROC 0.9834
Epoch 12: Train Loss 0.0776, Val AUROC 0.9824
Epoch 13: Train Loss 0.0737, Val AUROC 0.9837
Epoch 14: Train Loss 0.0674, Val AUROC 0.9846
Epoch 15: Train Loss 0.0633, Val AUROC 0.9841
Epoch 16: Train Loss 0.0616, Val AUROC 0.9863
Epoch 17: Train Loss 0.0610, Val AUROC 0.9854
Epoch 18: Train Loss 0.0565, Val AUROC 0.9852
Epoch 19: Train Loss 0.0537, Val AUROC 0.9878
Epoch 20: Train Loss 0.0518, Val AUROC 0.9876
Epoch 21: Train Loss 0.0492, Val AUROC 0.9865
Epoch 22: Train Loss 0.0476, Val AUROC 0.98

In [16]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set
model.eval()
test_preds = []
test_labels = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))

Accuracy: 0.957724405361188
accuracy another:  0.972651080723423


In [19]:
print(classification_report(test_labels, test_preds))

              precision    recall  f1-score   support

           0       0.92      0.93      0.93       411
           1       0.99      0.98      0.98      1856

    accuracy                           0.97      2267
   macro avg       0.95      0.96      0.95      2267
weighted avg       0.97      0.97      0.97      2267



In [ ]:
roc_auc_score(test_labels, test_preds)

0.9577244053611881

In [21]:
# Recreate the model architecture
model_1 = FineTunedMultimodalModel(num_layers_to_unfreeze=1)
model_1.load_state_dict(torch.load('best_finetuned_model.pth', map_location='cpu'))  # or 'cuda' if using GPU
model_1.to(device)
model_1.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at DeepChem/ChemBERTa-77M-MTR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FineTunedMultimodalModel(
  (smiles_encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(600, 384, padding_idx=1)
      (position_embeddings): Embedding(515, 384, padding_idx=1)
      (token_type_embeddings): Embedding(1, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.144, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-2): 3 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.109, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
          

In [22]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Run inference on test set

test_preds = []
test_labels = []
with torch.no_grad():
	for batch in test_loader:
		input_ids = batch['input_ids'].to(device)
		attention_mask = batch['attention_mask'].to(device)
		descriptors = batch['descriptors'].to(device)
		labels = batch['labels'].to(device)
		logits = model_1(input_ids, attention_mask, descriptors)
		probs = torch.sigmoid(logits).cpu().numpy()
		test_preds.extend((probs > 0.5).astype(int))
		test_labels.extend(labels.cpu().numpy())

accuracy = balanced_accuracy_score(test_labels, test_preds)
print(f"Accuracy: {accuracy}")
print("accuracy another: ", accuracy_score(test_labels, test_preds))
roc_auc = roc_auc_score(test_labels, test_preds)
print(f"ROC AUC: {roc_auc}")

Accuracy: 0.9560995049920296
accuracy another:  0.9730921923246582
ROC AUC: 0.9560995049920296
